In [ ]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import requests
from datetime import datetime
from typing import List, Union, Dict, Any
from datetime import timedelta

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
MODEL = "gpt-4.1-mini"
openai = OpenAI(api_key=openai_api_key)

This ia stock assistant. The tool queries marketstack API to get the past two weeks stock data. You have to get the free marketstack api key and put it in .env.
LLM can then use the data to perform analysis.

The tool function get_stock_data takes the stock symbol as parameter. LLM is able to pass the symbol even if the user asks for the companies name. This might not work if the companies that went on ipo recently.
The tool function get_stock_data is hard coded to get only past 2 weeks data.

Promtps
What is the peformance of Apple. 
Compare performances of Apple and microsoft and give me the best pick to invest in.


In [ ]:
system_message = """
You are a helpful Stock advisor. You good at giving information and advice related to particular stocks.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
You are capable of giving a report on a stock's performance in the past couple of weeks.
"""

In [ ]:



def get_stock_data(stock):
    print(f"STOCK TOOL CALLED: Getting price for {stock}", flush=True)
    date_from = (datetime.now() - timedelta(days=14)).strftime("%Y-%m-%d")
    date_to = datetime.now().strftime("%Y-%m-%d")
    return get_eod_marketstack(symbols=stock, date_from=date_from, date_to=date_to)
    

In [ ]:
get_stock_data("AAPL")

In [ ]:
def get_eod_marketstack(
    symbols: Union[str, List[str]], 
    date_from: str, 
    date_to: str,
    limit: int = 100
) -> Dict[str, Any]:
    """
    Fetch End-of-Day (EOD) stock data from marketstack API.
    
    Parameters:
    -----------
    symbols : Union[str, List[str]]
        Stock symbol(s) to fetch. Can be a single string or list of strings.
        Example: "AAPL" or ["AAPL", "GOOGL", "MSFT"]
    date_from : str
        Start date in format 'YYYY-MM-DD'
        Example: "2024-01-01"
    date_to : str
        End date in format 'YYYY-MM-DD'
        Example: "2024-12-31"
    limit : int, optional
        Number of records per page (default: 100)
    
    Returns:
    --------
    Dict[str, Any]
        API response containing EOD data
        
    Raises:
    -------
    ValueError
        If dates are invalid or API key is missing
    requests.exceptions.RequestException
        If API request fails
    """
    # Get API key from environment
    api_key = os.getenv('MARKETSTACK_API_KEY')
    if not api_key:
        raise ValueError("MARKETSTACK_API_KEY not found in environment variables")
    
    # Convert symbols list to comma-separated string if needed
    if isinstance(symbols, list):
        symbols_str = ",".join(symbols)
    else:
        symbols_str = symbols
    
    # Validate date format
    try:
        datetime.strptime(date_from, "%Y-%m-%d")
        datetime.strptime(date_to, "%Y-%m-%d")
    except ValueError as e:
        raise ValueError(f"Invalid date format. Use 'YYYY-MM-DD'. Error: {e}")
    
    # API endpoint
    url = "https://api.marketstack.com/v1/eod"
    
    # Request parameters
    params = {
        "access_key": api_key,
        "symbols": symbols_str,
        "date_from": date_from,
        "date_to": date_to,
        "limit": limit,
        "sort": "DESC"  # Most recent data first
    }
    
    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"Error fetching EOD data: {e}")
        raise

In [ ]:
get_eod_marketstack(symbols="AAPL", date_from="2026-03-26", date_to="2026-03-27")

In [ ]:
stock_function = {
    "name": "get_stock_data",
    "description": "Get the stock data for a specific company.",
    "parameters": {
        "type": "object",
        "properties": {
            "symbol": {
                "type": "string",
                "description": "The stock symbol of the company.",
            },
        },
        "required": ["symbol"],
        "additionalProperties": False
    }
}
tools = [{"type": "function", "function": stock_function}]

In [ ]:
def chat(history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history 
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    reply = response.choices[0].message.content
    history += [{"role":"assistant", "content":reply}]

    voice = talker(reply)
    
    return history, voice

In [ ]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_stock_data":
            arguments = json.loads(tool_call.function.arguments)
            symbol = arguments.get('symbol')
            stock_details = get_stock_data(symbol)
            responses.append({
                "role": "tool",
                "content": json.dumps(stock_details),
                "tool_call_id": tool_call.id
            })
    return responses

In [ ]:
def put_message_in_chatbot(message, history):
        return "", history + [{"role":"user", "content":message}]

# UI definition

with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=500, type="messages")
    with gr.Row():
        audio_output = gr.Audio(autoplay=True)
    with gr.Row():
        message = gr.Textbox(label="Chat with our AI Assistant:")

# Hooking up events to callbacks

    message.submit(put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]).then(
        chat, inputs=chatbot, outputs=[chatbot, audio_output]
    )

ui.launch(inbrowser=True)

In [ ]:
def talker(message):
    response = openai.audio.speech.create(
      model="gpt-4o-mini-tts",
      voice="onyx",    # Also, try replacing onyx with alloy or coral
      input=message
    )
    return response.content